In [4]:
import duckdb
import pandas as pd
import plotly.express as px

# Connect to our local warehouse
# Notebooks are in /notebooks/ and data is in /data/ (sibling directories)
db_path = '../data/logistics_warehouse.duckdb'
con = duckdb.connect(db_path)

# JOIN the Fact and Dimension tables (Standard SQL Practice)
query = """
SELECT 
    f.*, 
    d.service_name,
    d.base_rate_per_kg
FROM fact_shipments f
JOIN dim_services d ON f.service_id = d.service_id
"""
df = con.execute(query).df()

# Convert ship_date to datetime for time-series analysis
df['ship_date'] = pd.to_datetime(df['ship_date'])

In [ ]:
# Group by Region to see where the money is
regional_perf = df.groupby('region').agg({
    'revenue': 'sum',
    'profit': 'sum',
    'shipment_id': 'count'
}).reset_index()

# Calculate Profit Margin %
regional_perf['margin_pct'] = (regional_perf['profit'] / regional_perf['revenue']) * 100

print("--- Regional Performance Strategy ---")
print(regional_perf.sort_values(by='margin_pct', ascending=False))

# Visualization: Revenue vs Profit
fig = px.bar(regional_perf, x='region', y=['revenue', 'profit'], 
             barmode='group', title='Revenue vs Profit by Region (DHL Red Style)',
             color_discrete_map={'revenue': '#D40511', 'profit': '#FFCC00'})
fig.show()